# Lab 4: ML-Powered Customer Analytics

This notebook implements the Kaggle packaging, training-serving skew, parity, what-if, and metadata tasks from the handout. It uses the Week 3 model; it does not invent or retrain a replacement model.

Before running the lab cells, place `churn_model.joblib` and the original IBM Telco churn CSV in this notebook's working directory. The Week 3 scikit-learn version, CV AUC mean and standard deviation, test AUC, and cost-based threshold are also required for truthful metadata. The current runtime reported that the model and CSV are not present, so result cells will remain safely skipped until those inputs are supplied.

The handout's Week 3 encoder uses `get_dummies(drop_first=True)`. If your Week 3 notebook used additional preprocessing, use its original feature-building function in the marked cell and let the parity assertion verify the match.

In [9]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

RAW_COLS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges",
]

WORK_DIR = Path.cwd()
MODEL_PATH = WORK_DIR / "churn_model.joblib"
DATA_PATH = WORK_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print(f"Working directory: {WORK_DIR}")
print(f"scikit-learn in this runtime: {sklearn.__version__}")
print(f"Model artifact present: {MODEL_PATH.exists()}")
print(f"Telco CSV present: {DATA_PATH.exists()}")

Working directory: /content
scikit-learn in this runtime: 1.6.1
Model artifact present: False
Telco CSV present: False


In [10]:
def build_week3_features(raw):
    """Reproduce the handout's Week 3 drop-first training encoder."""
    cleaned = raw[RAW_COLS].copy()
    cleaned["TotalCharges"] = pd.to_numeric(
        cleaned["TotalCharges"], errors="coerce"
    ).fillna(0)
    return pd.get_dummies(cleaned, drop_first=True).astype(float)


def prepare_input(raw, columns):
    """Encode raw customer rows with the serving-time, no-drop-first scheme."""
    missing = [column for column in RAW_COLS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing raw customer columns: {missing}")
    cleaned = raw[RAW_COLS].copy()
    cleaned["TotalCharges"] = pd.to_numeric(
        cleaned["TotalCharges"], errors="coerce"
    ).fillna(0)
    encoded = pd.get_dummies(cleaned)
    return encoded.reindex(columns=columns, fill_value=0).astype(float)


if MODEL_PATH.is_file() and DATA_PATH.is_file():
    df = pd.read_csv(DATA_PATH)
    if "Churn" not in df.columns:
        raise ValueError("The input CSV must include the Churn target column.")
    model = joblib.load(MODEL_PATH)
    cols = list(getattr(model, "feature_names_in_", []))
    if not cols:
        raise ValueError(
            "The loaded model has no feature_names_in_. Use the exact ordered "
            "training columns from the Week 3 notebook in `cols`."
        )

    raw = df.drop(columns=["Churn"])
    X = build_week3_features(raw).reindex(columns=cols, fill_value=0)
    final = model[-1] if hasattr(model, "steps") else model

    print("Task 1.1: Model packaging")
    print("Model:", type(final).__name__)
    print("Feature columns:", len(cols))
    print("Runtime scikit-learn:", sklearn.__version__)

    two_year = raw.loc[raw["Contract"].eq("Two year")].head(1)
    if two_year.empty:
        raise ValueError("No Two year contract row exists in the supplied CSV.")
    wrong = build_week3_features(two_year).reindex(columns=cols, fill_value=0)
    right = prepare_input(two_year, cols)
    p_wrong = float(model.predict_proba(wrong)[0, 1])
    p_right = float(model.predict_proba(right)[0, 1])
    print("\nTask 1.3: Training-serving skew demo")
    print("Contract columns, wrong:", wrong.filter(like="Contract").values)
    print("Contract columns, right:", right.filter(like="Contract").values)
    print(f"P(churn) wrong: {p_wrong:.3f}")
    print(f"P(churn) right: {p_right:.3f}")

    p_train = model.predict_proba(X[cols])[:, 1]
    p_serve = model.predict_proba(prepare_input(raw, cols))[:, 1]
    parity_difference = float(np.abs(p_train - p_serve).max())
    print("\nTask 1.4: All-customer parity test")
    print("Largest difference:", parity_difference)
    assert np.allclose(p_train, p_serve), "Training-serving skew detected!"
    print(f"Parity test passed on {len(raw)} customers")

    def what_if(model, row, columns, changes):
        baseline = float(model.predict_proba(prepare_input(row, columns))[0, 1])
        results = []
        for feature, value in changes:
            alternative = row.copy()
            alternative.loc[:, feature] = value
            probability = float(
                model.predict_proba(prepare_input(alternative, columns))[0, 1]
            )
            results.append(
                {
                    "change": f"{feature} = {value}",
                    "p_churn": round(probability, 3),
                    "delta": round(probability - baseline, 3),
                }
            )
        return baseline, pd.DataFrame(results)

    riskiest_index = int(np.argmax(p_serve))
    what_if_row = raw.iloc[[riskiest_index]].copy()
    baseline, what_if_table = what_if(
        model,
        what_if_row,
        cols,
        [
            ("Contract", "One year"),
            ("Contract", "Two year"),
            ("PaymentMethod", "Credit card (automatic)"),
        ],
    )
    print("\nTask 1.5: What-if analysis")
    print(f"Baseline P(churn) = {baseline:.3f}")
    display(what_if_table)
    lowest_risk_change = what_if_table.loc[what_if_table["delta"].idxmin()]
    print(
        f"Largest modeled reduction: {lowest_risk_change['change']} "
        f"(delta {lowest_risk_change['delta']:+.3f}); this is an association, "
        "not evidence that the change causes retention."
    )
else:
    missing_files = [
        str(path) for path in (MODEL_PATH, DATA_PATH) if not path.is_file()
    ]
    print("Task 1 is ready but needs the Week 3 inputs:")
    print("\n".join(f"- {path}" for path in missing_files))
    print(
        "Upload the model and original Telco CSV, or update MODEL_PATH and "
        "DATA_PATH in the setup cell. Then rerun this cell."
    )

Task 1 is ready but needs the Week 3 inputs:
- /content/churn_model.joblib
- /content/WA_Fn-UseC_-Telco-Customer-Churn.csv
Upload the model and original Telco CSV, or update MODEL_PATH and DATA_PATH in the setup cell. Then rerun this cell.


In [15]:
# Enter the real values printed by Week 3 and chosen in the Week 2 cost analysis.
TRAINING_SKLEARN_VERSION = None
CV_AUC_MEAN = None
CV_AUC_STD = None
TEST_AUC = None
THRESHOLD = None

metric_values = {
    "TRAINING_SKLEARN_VERSION": TRAINING_SKLEARN_VERSION,
    "CV_AUC_MEAN": CV_AUC_MEAN,
    "CV_AUC_STD": CV_AUC_STD,
    "TEST_AUC": TEST_AUC,
    "THRESHOLD": THRESHOLD,
}
missing_metrics = [name for name, value in metric_values.items() if value is None]
parity_passed = (
    "p_train" in globals()
    and "p_serve" in globals()
    and np.allclose(p_train, p_serve)
)
meta = None

if not parity_passed:
    print("Metadata is blocked until the all-customer parity test passes.")
elif not all(name in globals() for name in ("df", "raw", "cols", "final")):
    print("Metadata will be saved after the Week 3 model and data are loaded.")
elif missing_metrics:
    print("Enter the real Week 2/Week 3 values before saving metadata:")
    print("\n".join(f"- {name}" for name in missing_metrics))
else:
    meta = {
        "model_name": type(final).__name__,
        "sklearn_version": str(TRAINING_SKLEARN_VERSION),
        "feature_columns": cols,
        "threshold": float(THRESHOLD),
        "cv_auc": float(CV_AUC_MEAN),
        "cv_auc_std": float(CV_AUC_STD),
        "test_auc": float(TEST_AUC),
        "training_data": "IBM Telco Customer Churn, 7,043 customers",
        "version": "1.0",
    }
    metadata_path = WORK_DIR / "model_meta.json"
    sample_path = WORK_DIR / "sample_customers.csv"
    metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    raw.sample(50, random_state=1).to_csv(sample_path, index=False)
    print(f"Saved {metadata_path} with {len(cols)} ordered feature columns")
    print(f"Saved {sample_path} with 50 raw customer rows")

Metadata is blocked until the all-customer parity test passes.


In [12]:
%%writefile app.py
import json

import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Churn Risk Advisor", layout="wide")

RAW_COLS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges",
]
SERVICES = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies",
]
PAYMENTS = [
    "Electronic check", "Mailed check", "Bank transfer (automatic)",
    "Credit card (automatic)",
]


@st.cache_resource
def load_artifacts():
    model = joblib.load("churn_model.joblib")
    with open("model_meta.json", encoding="utf-8") as file:
        meta = json.load(file)
    return model, meta


def prepare_input(raw, columns):
    missing = [column for column in RAW_COLS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing raw customer columns: {missing}")
    cleaned = raw[RAW_COLS].copy()
    cleaned["TotalCharges"] = pd.to_numeric(
        cleaned["TotalCharges"], errors="coerce"
    ).fillna(0)
    encoded = pd.get_dummies(cleaned)
    return encoded.reindex(columns=columns, fill_value=0).astype(float)


model, meta = load_artifacts()
COLS = meta["feature_columns"]


def score(raw):
    return model.predict_proba(prepare_input(raw, COLS))[:, 1]


with st.sidebar:
    st.header("Customer profile")
    tenure = st.slider("Tenure (months)", 0, 72, 4)
    contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    monthly = st.number_input("Monthly charges (USD)", 18.0, 120.0, 85.0, step=0.5)
    internet = st.selectbox("Internet service", ["Fiber optic", "DSL", "No"])
    services = []
    if internet != "No":
        services = st.multiselect("Add-on services", SERVICES, default=["StreamingTV"])
    phone = st.radio("Phone service", ["Yes", "No"], horizontal=True)
    multi = "No phone service"
    if phone == "Yes":
        multi = st.radio("Multiple lines", ["No", "Yes"], horizontal=True)
    payment = st.selectbox("Payment method", PAYMENTS)
    paperless = st.radio("Paperless billing", ["Yes", "No"], horizontal=True)
    with st.expander("Demographics"):
        gender = st.radio("Gender", ["Female", "Male"], horizontal=True)
        senior = st.checkbox("Senior citizen")
        partner = st.checkbox("Has partner")
        dependents = st.checkbox("Has dependents")
    st.divider()
    threshold = st.slider(
        "Contact threshold", 0.05, 0.95, float(meta["threshold"]), 0.05
    )

row = {
    "gender": gender,
    "SeniorCitizen": int(senior),
    "Partner": "Yes" if partner else "No",
    "Dependents": "Yes" if dependents else "No",
    "tenure": tenure,
    "PhoneService": phone,
    "MultipleLines": multi,
    "InternetService": internet,
    "Contract": contract,
    "PaperlessBilling": paperless,
    "PaymentMethod": payment,
    "MonthlyCharges": monthly,
    "TotalCharges": tenure * monthly,
}
for service in SERVICES:
    row[service] = (
        "No internet service" if internet == "No"
        else "Yes" if service in services
        else "No"
    )
customer = pd.DataFrame([row])

st.title("Customer Churn Risk Advisor")
st.caption(
    f"{meta['model_name']}  |  CV AUC {meta['cv_auc']:.3f} "
    f"+/- {meta['cv_auc_std']:.3f}  |  Decision support only"
)
tab1, tab2, tab3 = st.tabs(["One customer", "Batch scoring", "About"])

with tab1:
    probability = float(score(customer)[0])
    band = (
        "HIGH" if probability >= threshold
        else "WATCH" if probability >= threshold / 2
        else "LOW"
    )
    first, second, third = st.columns(3)
    first.metric("Churn probability", f"{probability:.0%}")
    second.metric("Risk band", band)
    third.metric("Action", "Contact now" if band == "HIGH" else "No action")
    st.progress(probability)

    st.subheader("What would change the risk?")
    changes = [
        ("Contract", option)
        for option in ["One year", "Two year"]
        if option != contract
    ]
    if payment == "Electronic check":
        changes.append(("PaymentMethod", "Credit card (automatic)"))
    if internet != "No" and "TechSupport" not in services:
        changes.append(("TechSupport", "Yes"))
    what_if_rows = []
    for feature, value in changes:
        alternative = customer.copy()
        alternative.loc[:, feature] = value
        new_probability = float(score(alternative)[0])
        what_if_rows.append(
            {
                "Change": f"{feature}: {value}",
                "New probability": round(new_probability, 3),
                "Difference": round(new_probability - probability, 3),
            }
        )
    if what_if_rows:
        st.dataframe(pd.DataFrame(what_if_rows), hide_index=True)
    st.caption("Associations learned from data, not guaranteed effects.")

    final_model = model[-1] if hasattr(model, "steps") else model
    if hasattr(model, "steps") and hasattr(final_model, "coef_"):
        with st.expander("Why this score? (Logistic Regression)"):
            transformed = model[:-1].transform(prepare_input(customer, COLS))[0]
            contributions = pd.Series(final_model.coef_[0] * transformed, index=COLS)
            largest = contributions.reindex(
                contributions.abs().sort_values(ascending=False).index
            ).head(8)
            st.bar_chart(largest)
            st.caption("Positive bars push toward churn, negative away.")

with tab2:
    st.write("Upload a CSV with the original Telco columns.")
    uploaded_file = st.file_uploader("Customer file", type="csv")
    if uploaded_file is not None:
        data = pd.read_csv(uploaded_file)
        missing = [column for column in RAW_COLS if column not in data.columns]
        if missing:
            st.error(f"Missing columns: {missing}")
            st.stop()
        data["TotalCharges"] = pd.to_numeric(
            data["TotalCharges"], errors="coerce"
        ).fillna(0)
        data["p_churn"] = score(data).round(3)
        data["contact"] = np.where(data["p_churn"] >= threshold, "Yes", "")
        contacts = int((data["contact"] == "Yes").sum())
        st.write(f"{contacts} of {len(data)} customers are above the threshold.")
        st.dataframe(
            data.sort_values("p_churn", ascending=False).head(50),
            hide_index=True,
        )
        st.download_button(
            "Download scored CSV",
            data.to_csv(index=False),
            "scored_customers.csv",
            "text/csv",
        )

with tab3:
    st.json({key: value for key, value in meta.items() if key != "feature_columns"})
    st.markdown(
        "**Limitations:** trained on one US telecom dataset; not validated "
        "for other markets. See the model card."
    )

Writing app.py


In [13]:
app_path = WORK_DIR / "app.py"
app_source = app_path.read_text(encoding="utf-8")
compile(app_source, str(app_path), "exec")
print(f"app.py syntax is valid: {app_path}")

if TRAINING_SKLEARN_VERSION is None:
    print("requirements.txt is waiting for the exact Week 3 scikit-learn version.")
else:
    requirements = (
        "streamlit>=1.32\n"
        f"scikit-learn=={TRAINING_SKLEARN_VERSION}\n"
        "pandas\n"
        "numpy\n"
        "joblib\n"
    )
    requirements_path = WORK_DIR / "requirements.txt"
    requirements_path.write_text(requirements, encoding="utf-8")
    print(f"Saved {requirements_path}")

app.py syntax is valid: /content/app.py
requirements.txt is waiting for the exact Week 3 scikit-learn version.


In [16]:
# Add these personal project details after deployment; no placeholders are written as final claims.
OWNER_NAME = None
PUBLIC_URL = None
WEEK1_TO_3_NOTEBOOK_URL = None
SINGLE_CUSTOMER_SCREENSHOT = None
BATCH_SCREENSHOT = None

personal_fields = {
    "OWNER_NAME": OWNER_NAME,
    "PUBLIC_URL": PUBLIC_URL,
    "WEEK1_TO_3_NOTEBOOK_URL": WEEK1_TO_3_NOTEBOOK_URL,
    "SINGLE_CUSTOMER_SCREENSHOT": SINGLE_CUSTOMER_SCREENSHOT,
    "BATCH_SCREENSHOT": BATCH_SCREENSHOT,
}
missing_personal_fields = [
    name for name, value in personal_fields.items() if not value
]
missing_screenshots = [
    str(value)
    for value in (SINGLE_CUSTOMER_SCREENSHOT, BATCH_SCREENSHOT)
    if value and not Path(value).is_file()
]

if not isinstance(globals().get("meta"), dict):
    print("README generation is waiting for a passing parity test and model metadata.")
elif missing_personal_fields or missing_screenshots:
    print("Provide these real project details before generating README.md:")
    print("\n".join(f"- {name}" for name in missing_personal_fields + missing_screenshots))
else:
    readme = f"""# Churn Risk Advisor

## Model card: Churn Risk Advisor v1.0

**Intended use:** Rank existing telecom customers by churn risk so a retention team can prioritize calls. Decision support, not automation.
**Not for:** Credit, pricing, or any decision that denies a service.
**Data:** IBM Telco Customer Churn, 7,043 customers, 26.5% churn.
**Model:** {meta['model_name']}, chosen by 5-fold CV. Features: {len(meta['feature_columns'])} encoded columns.
**Performance:** CV AUC {meta['cv_auc']:.3f} +/- {meta['cv_auc_std']:.3f}; test AUC {meta['test_auc']:.3f} (test set used once).
**Threshold:** {meta['threshold']:.2f}, from the Week 2 cost analysis (FN = PKR 6,000, FP = PKR 1,000).
**Limitations:** One US dataset from one period; no Pakistan data; associations, not causes; performance may drift as plans change.
**Fairness check (Week 15):** Compare recall across gender and senior-citizen groups.
**Owner and version:** {OWNER_NAME}, v1.0.

## Live app

{PUBLIC_URL}

## Screenshots

![Single customer view]({SINGLE_CUSTOMER_SCREENSHOT})

![Batch scoring view]({BATCH_SCREENSHOT})

## Run locally

```powershell
python -m venv .venv
.venv\\Scripts\\activate
pip install -r requirements.txt
streamlit run app.py
```

[Week 1-3 notebooks]({WEEK1_TO_3_NOTEBOOK_URL})
"""
    readme_path = WORK_DIR / "README.md"
    readme_path.write_text(readme, encoding="utf-8")
    print(f"Saved {readme_path}")

README generation is waiting for a passing parity test and model metadata.


## Check Your Understanding

1. `get_dummies(drop_first=True)` chooses the dropped category from the categories present in the input batch. For a single row, the only observed contract can be dropped, leaving all contract indicators at zero; after reindexing, the model can mistake it for the training baseline.
2. The parity test compares probabilities from the Week 3 training representation and the serving encoder for every training customer. Testing all rows checks all observed categories and combinations instead of relying on a sample that might miss a skew case.
3. Use `st.cache_resource` for the loaded model because it is a long-lived, shared resource. `st.cache_data` is for cached data outputs, not model instances.
4. The app derives `TotalCharges` to avoid asking for a value that should be consistent with tenure and monthly charges and is easy to mistype. The estimate is still approximate because real billing can include discounts, partial months, and other adjustments.
5. A serialized scikit-learn model depends on compatible estimator and preprocessing implementations. Pinning the exact training version makes the deployed environment reproducible and avoids version-related loading or prediction differences.
6. A lower what-if probability is an association learned from historical data, not proof that offering a two-year contract causes retention. A company could test the offer with a randomized experiment and compare retention rates while monitoring customer impact.

## Remaining Personal Deliverables

The notebook creates `app.py` and will create `requirements.txt`, `model_meta.json`, and `sample_customers.csv` after the real training files and values are supplied. The public deployment URL, screenshots, backup demo video, named presenter/owner, Week 1-3 notebook link, fairness comparison, and LinkedIn post require your account, project results, and personal materials; they are not filled with invented values.

## Presentation 1: Five-Minute Draft

Replace every bracket with a result from the completed notebook before presenting. Keep the live app demo between slides 3 and 4, and record the 60-second backup demo after the app works.

1. **Problem, 0:00-0:40:** “A churned customer costs about PKR 6,000 a year.” State the business problem and source for the cost.
2. **Data, 0:40-1:20:** “Churn risk is concentrated in [measured segment].” Use one chart and your calculated segment rates.
3. **Model and evidence, 1:20-2:20:** “The model ranks risk with [CV AUC +/- standard deviation].” State the test AUC, cost-based threshold, parity result, and one limitation.
4. **Live demo, 2:20-3:50:** Show one high-risk customer, the probability and risk band, one what-if comparison, and batch scoring.
5. **Action and lesson, 3:50-5:00:** “We should test [retention action] with [segment].” Explain that what-if scores are associations, show the public URL, and answer questions honestly.

Do not present the example numbers or segment claims above as measured results. Add your real outputs, app URL, and named segment after the Week 3 artifacts are available.